In [1]:
import os
import re
import pandas as pd
import logging
from underthesea import chunk
from sklearn.model_selection import train_test_split
import ast
import time
from gensim import corpora, models

In [2]:
QUERY_DATASET_PATH = "dataset\\restaurant_dataset_ver1.csv"
DATASET_PATH = "dataset\\foody_combined_data_final.csv"
QUERY_TEXT_PATH = "dataset\\restaurant_queries.txt"
base_path = os.getcwd()

# To be change if needed
actual_query_dataset_path = os.path.join(base_path, "..", "..", QUERY_DATASET_PATH)
actual_dataset_path = os.path.join(base_path, "..", "..", DATASET_PATH)
actual_query_text_path = os.path.join(base_path, "..", "..", QUERY_TEXT_PATH)

In [3]:
stopwords = []
with open("stopwords.txt", 'r', encoding='utf-8') as f:
    stopwords = f.read().split()


In [4]:
time_related_words = ["hôm nay", "hôm qua", "ngày mai", "tuần này", "tuần sau", "tháng này", "tháng sau", "năm nay", "năm sau"] 
time_of_day_related_words = ["sáng", "trưa", "chiều", "tối", "đêm"]
time_of_week_related_words = ["thứ 2", "thứ 3", "thứ 4", "thứ 5", "thứ 6", "thứ 7", "chủ nhật", "thứ hai", "thứ ba", "thứ tư", "thứ năm", "thứ sáu", "thứ bảy"]
time_amount_related_words = ["h", 'giờ', 'phút', 'p']

location_related_words = ["quận", "huyện", "phường", "xã", "thị trấn", "tỉnh", "thành phố", "đường", "phố", "ngõ", "hẻm"]
good_attribute_words = ["tốt", "ngon", "đẹp", "rẻ", "vui vẻ", "thân thiện", "sạch sẽ", "đáng tiền", "hợp lý"]
invert_attribute_words = ["không", "chẳng", "chả", "chưa", "chẳng phải", "không phải"]
amount_related_words = ["nhiều", "ít", "đầy đủ", "vừa đủ", "quá nhiều", "quá ít", "phải chăng", ]
pronoun_related_words = ["tôi", "tao", "mày", "cậu", "chúng tôi", "chúng ta", "họ", "bạn", "các bạn"]
intro_related_words = ["giới thiệu", "tìm kiếm", "tìm", "tôi muốn tìm", "tôi muốn giới thiệu", "tôi muốn tìm kiếm", "tôi muốn tìm hiểu về", "tôi muốn biết về", "hãy", "gợi ý"]
misc_words = ['giúp', 'cho', 'nhờ', 'về', 'có', 'là', 'được', 'cũng', 'nên', 'cần', 'phải', 'muốn', 'thích', 'yêu thích', 'ưa thích', 'chỗ']

limit_related_words = ["giới hạn", "hạn chế", "tối đa", "tối thiểu", "ít nhất", "nhiều nhất", "không quá", "không dưới"]

service_related_words = ["phục vụ", "đón tiếp", "hỗ trợ", "tư vấn", "giao hàng", "đặt bàn", "đặt món"]
puncuation_pattern = r'[^\w\s]'

purpose_related_words = ["ăn uống", "đi chơi", "hẹn hò", "gặp gỡ bạn bè", "tổ chức sự kiện", "họp mặt gia đình", "làm việc", "học tập", "thư giãn", "giải trí", "ăn vặt", "ăn gia đình", "hẹn hò", "bbq - món nướng", "họp nhóm", "đãi tiệc", "ăn chay", "giao tận nơi", "uống bia - nhậu", "hát hò", "chụp hình", "quay phim", "tiếp khách", "ăn bình dân", "quán cóc", "buffet", "takeaway", "mang về", "tổ chức tiệc cưới", "ngẵm cảnh", "thư giản", "đọc sách", "học bài", "nghe nhạc", "du lịch", "ăn fastfood", "tiếp khách sang trọng", "dành cho nam giới", "dã ngoại", "tiệc ngoài trời", "xem phim hd", "tổ chức hội thảo", "tiệc tận nơi", "mua sắm", "ăn chơi ngày tết"]
synonym_of_purpose_related_words = {
    "ăn uống": ["ăn uống", "ăn", "uống"],
    "đi chơi": ["đi chơi", "chơi", "vui chơi"],
    "hẹn hò": ["hẹn hò", "hẹn hẹn", "hò hẹn"],
    "hát hò": ["hát hò", "hát", "hò hò"],
    "chụp hình": ["chụp hình", "chụp ảnh"],
    "quay phim": ["quay phim", "quay video"],
    "tiếp khách": ["tiếp khách", "tiếp đãi", "tiếp"],
    "ăn bình dân": ["ăn bình dân", "ăn dân dã", "ăn bình dân"],
    "quán cóc": ["quán cóc", "quán vỉa hè", "quán lề đường"],
    "buffet": ["buffet", "ăn buffet", "tiệc buffet"],
    "takeaway": ["takeaway", "mang đi", "mang về"],
    "tổ chức tiệc cưới": ["tổ chức tiệc cưới", "tổ chức đám cưới", "tổ chức lễ cưới"],
    "ngắm cảnh": ["ngắm cảnh", "ngắm", "ngắm nhìn"],
    "thư giãn": ["thư giãn", "nghỉ ngơi", "giải lao", "giải trí"],
    "đọc sách": ["đọc sách", "đọc", "đọc truyện"],
    "học bài": ["học bài", "học", "học tập"],
    "nghe nhạc": ["nghe nhạc", "nghe", "nghe nhạc"],
    "du lịch": ["du lịch", "tham quan", "khám phá", "ngoại du"],
    "ăn fastfood": ["ăn fastfood", "ăn nhanh", "ăn nhanh"],
    "tiếp khách sang trọng": ["tiếp khách sang trọng", "tiếp khách cao cấp", "tiếp khách sang"],
    "dành cho nam giới": ["dành cho nam giới", "cho nam giới", "dành cho phái mạnh"],
    "dã ngoại": ["dã ngoại", "ngoài trời", "thăm quan ngoài trời"],
    "tiệc ngoài trời": ["tiệc ngoài trời", "tiệc sân vườn", "tiệc ngoài"],
    "xem phim hd": ["xem phim hd", "xem phim", "phim hd", "phim"],
    "tổ chức hội thảo": ["tổ chức hội thảo", "tổ chức seminar", "tổ chức workshop"],
    "tiệc tận nơi": ["tiệc tận nơi", "tiệc tại nhà", "tiệc tại chỗ"],
    "gặp gỡ bạn bè": ["gặp gỡ bạn bè", "gặp gỡ", "gặp mặt", "gặp"],
    "tổ chức sự kiện": ["tổ chức sự kiện", "tổ chức event", "tổ chức buổi tiệc"],
    "họp mặt gia đình": ["họp mặt gia đình", "họp mặt", "gặp gỡ gia đình"],
    "ăn vặt": ["ăn vặt", "ăn nhẹ", "ăn uống nhẹ"],
    "ăn gia đình": ["ăn gia đình", "ăn cùng gia đình", "ăn tại nhà"],
    "bbq - món nướng": ["bbq - món nướng", "nướng", "bbq"],
    "họp nhóm": ["họp nhóm", "họp mặt nhóm", "gặp gỡ nhóm"],
    "đãi tiệc": ["đãi tiệc", "đón tiếp", "tiệc tùng"],
    "ăn chay": ["ăn chay", "chay", "thực phẩm chay"],
    "giao tận nơi": ["giao tận nơi", "giao hàng tận nơi", "giao thức ăn"],
    "uống bia - nhậu": ["uống bia - nhậu", "uống bia", "nhậu"]
}
currency_related_words = ["đồng", "nghìn", "triệu", "vnd", "vnđ","k"]

cuisine_related_words = ["món", "món ăn", "ẩm thực", "đặc sản", "món truyền thống"]



In [5]:
def preprocess_line(line: list[tuple[str,str,str]]):
    remove_set = set(pronoun_related_words + stopwords + intro_related_words + misc_words)
    # remove all stopwords in line
    for word in remove_set:           
        line = [w for w in line if w[0] != word and re.search(puncuation_pattern, w[0]) is None]
        
    return line
def get_texts(sentence: list[tuple[str,str,str]]):
    return [word[0] for word in sentence]

In [6]:
query_dataset = pd.read_csv(actual_query_dataset_path)
X = query_dataset[["query", "restaurant_id"]]
y = query_dataset['label']
with open(actual_query_text_path, 'r', encoding='utf-8') as f:
    queries = f.read().splitlines()
    
train_queries, test_queries = train_test_split(queries, test_size=0.2, random_state=42)
X_train = X[X['query'].isin(train_queries)]
X_test = X[X['query'].isin(test_queries)]
y_train = y[X['query'].isin(train_queries)]
y_test = y[X['query'].isin(test_queries)]


In [7]:
store_meta_keywords_file_name = os.path.join(base_path, "..", "..", "tmp", "store_infos.csv")
if os.path.exists(store_meta_keywords_file_name):
    store_infos = pd.read_csv(store_meta_keywords_file_name, encoding='utf-8')
    store_infos = store_infos['MetaKeywords'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
else:
    data = pd.read_csv(DATASET_PATH)
    store_infos = data['MetaKeywords'].apply(lambda x: x.lower() if isinstance(x, str) else x)
    store_infos = store_infos.apply(lambda x: chunk(x) if isinstance(x, str) else x)
    store_infos.to_csv(f'{store_meta_keywords_file_name}', index=False, encoding='utf-8')
    

In [8]:
preprocessed_store_infos_file_path = os.path.join(base_path, "..", "..", "tmp", "preprocessed_store_infos.csv")
if os.path.exists(preprocessed_store_infos_file_path):
    preprocessed_store_infos = pd.read_csv(preprocessed_store_infos_file_path, encoding='utf-8')
    preprocessed_store_infos = preprocessed_store_infos['MetaKeywords'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
else:
    store_infos.dropna(inplace=True)
    preprocessed_store_infos = store_infos.apply(preprocess_line).apply(get_texts)
    preprocessed_store_infos.to_csv(f'{preprocessed_store_infos_file_path}', index=False, encoding='utf-8')


In [9]:
model_path = os.path.join(base_path, "..", "..", "tmp", "word2vec.model")
word2vec_model = models.Word2Vec(preprocessed_store_infos, vector_size=200, min_count=2, workers=4, seed=42)
word2vec_model.save(model_path)

In [10]:
set_of_words = set()
for line in preprocessed_store_infos:
    set_of_words.update(set(list(line)))

In [11]:

def preprocess_for_word2vec(query: str, word2vec_model: models.Word2Vec):
    parsed_query:list[tuple[str,str,str]] = chunk(query.lower())
    parsed_query = preprocess_line(parsed_query) 
    new_query = []
    for word in parsed_query:
        if word in word2vec_model.wv.index_to_key:
            new_query.append(word)
    return get_texts(new_query)

In [12]:
def find_location(text: str):
    for word in location_related_words:
        r = re.search(r'\b' + word + r'\b', text)
        if r:
            return r.start()
    return -1

def extract_locations(sentence: list[tuple[str,str,str]]):
    all_location = []
    for i, word in enumerate(sentence):
        r = find_location(word[0])
        if r != -1:
            # true_location = word[0] + f" {sentence[i + 1][0]}"
            all_location.append(i)
    return all_location

def extract_time(sentence: list[tuple[str,str,str]]):
    all_time = []
    for i, word in enumerate(sentence):
        for tow in time_of_week_related_words + time_of_day_related_words:
            r = re.search(r'\b' + tow + r'\b', word[0])
            if r:
                all_time.append(i)
        for taw in time_amount_related_words:
            r = re.search(r'\b' + taw + r'\b', word[0])
            if r and i >= 2:
                all_time.append(i)
    return all_time

def extract_service(sentence: list[tuple[str,str,str]]):
    all_service = []
    for i, word in enumerate(sentence):
        for service in service_related_words:
            
            r = re.search(r'\b' + service + r'\b', word[0])
            if r:
                all_service.append(word[0])
    return all_service


data = pd.read_csv(actual_dataset_path)
def remove_empty(x):
    while '' in x:
        x.remove('')
    return x

cuisine_values = data["Cuisines"].str.lower().str.split('|')
cuisine_values.dropna(inplace=True)
cuisine_values = cuisine_values.apply(remove_empty)

cuisine_related_words = sorted(set(cuisine_related_words).union({
    cuisine.strip()
    for cuisines in cuisine_values
    for cuisine in cuisines
    if cuisine.strip()
}))

t = data["LstTargetAudience"].str.lower().str.split('|')
t.dropna(inplace=True)
t.apply(remove_empty)
dictionary = corpora.Dictionary(t)

dictionary.save(os.path.join(base_path, "..", "..", "tmp", "targetaudience_corpus.dict"))

def extract_for_who(sentence: list[tuple[str,str,str]]):
    all_target_score = []
    for i, word in enumerate(sentence):
        if word[0] in dictionary.token2id.keys():
            all_target_score.append(i)
    return all_target_score


def extract_cuisine(sentence: list[tuple[str, str, str]]):
    all_result = []
    for i, word in enumerate(sentence):
        for cuisine in cuisine_related_words:
            r = re.search(r'\b' + cuisine + r'\b', word[0])
            if r:
                all_result.append(i)
    return all_result

def to_minutes(time_point: str):
    if time_point is None:
        return -1
    hour, minute = time_point.split(':')
    return int(hour) * 60 + int(minute)

def parse_time_point(time_range: str):
    # DB time parse
    time_range_obj = ast.literal_eval(time_range)
    return (to_minutes(time_range_obj[0][0]), to_minutes(time_range_obj[0][1]))


def parse_price_tokens(sentence: list[str]):
    price_values = []
    seen_prices = set()
    multiplier_units = {"k", "nghìn", "nghin"}

    for i, token in enumerate(sentence):
        cleaned_token = token.strip().lower()
        if not cleaned_token:
            continue

        price_value = None
        if cleaned_token.endswith('k') and cleaned_token[:-1].isdigit():
            price_value = int(cleaned_token[:-1]) * 1000
        elif cleaned_token.isdigit():
            price_value = int(cleaned_token)
            if i + 1 < len(sentence):
                next_token = sentence[i + 1].strip().lower()
                if next_token in multiplier_units:
                    price_value *= 1000
        elif re.fullmatch(r'\d{1,3}(?:\.\d{3})+', cleaned_token):
            price_value = int(cleaned_token.replace('.', ''))

        if price_value is not None and price_value not in seen_prices:
            seen_prices.add(price_value)
            price_values.append(price_value)

    return price_values
    

In [13]:
if not os.path.exists(os.path.join(base_path, "..", "..", "tmp", "attribute model")):
    os.makedirs(os.path.join(base_path, "..", "..", "tmp", "attribute model"), exist_ok=True)

cuisine_dictionary = corpora.Dictionary([cuisine_related_words])
cuisine_model_bm25 = models.OkapiBM25Model(dictionary=cuisine_dictionary)
cuisine_model_bm25.save(os.path.join(base_path, "..", "..", "tmp", "attribute model", "cuisine_bm25.model"))

t_model_bm25 = models.OkapiBM25Model(dictionary.doc2bow(text) for text in t)
t_model_bm25.save(os.path.join(base_path, "..", "..", "tmp", "attribute model", "targetaudience_bm25.model"))

service_dictionary = corpora.Dictionary([service_related_words])
service_model_bm25 = models.OkapiBM25Model(dictionary=service_dictionary)
service_model_bm25.save(os.path.join(base_path, "..", "..", "tmp", "attribute model", "service_bm25.model"))

def time_prepare(sentence: list[tuple[str, str, str]], locations):
    text = get_texts(sentence)
    if len(locations) == 0:
        return []
    selected_day = []
    selected_time = []
    for p in locations:
        if text[p] in time_amount_related_words and p >= 2 and \
           time.strptime(text[p-2] + ":" + text[p -1], '%H:%M'):
            t = to_minutes(text[p-2] + ":" + text[p -1])
            selected_time.append(t)
        elif text[p] in time_of_week_related_words:
            selected_day.append(text[p])
    
    selected_values = {}
    for day, sel_time in zip(selected_day, selected_time):
        selected_values[day] = sel_time
    return selected_values

def embedding(sentence: list[tuple[str, str, str]], locations, model, dictionary):
    all_result = []
    for p in locations:
        word = sentence[p]
        all_result.append(model[dictionary.doc2bow([word[0]])])
    return all_result


### Tọa độ thập phân (Decimal Degrees)

| Khu vực | Vĩ độ (lat) | Kinh độ (lon) |
|---|---:|---:|
| Quận 1 | 10.776111 | 106.695833 |
| Quận 2 | 10.778611 | 106.757500 |
| Quận 3 | 10.780000 | 106.679444 |
| Quận 4 | 10.761667 | 106.702500 |
| Quận 5 | 10.756667 | 106.666667 |
| Quận 6 | 10.746111 | 106.636111 |
| Quận 7 | 10.738611 | 106.726389 |
| Quận 8 | 10.723333 | 106.627778 |
| Quận 9 | 10.840000 | 106.770833 |
| Quận 10 | 10.773611 | 106.667222 |
| Quận 11 | 10.766944 | 106.645556 |
| Quận 12 | 10.861944 | 106.658889 |
| Bình Thạnh | 10.802778 | 106.696667 |
| Tân Bình | 10.803611 | 106.650833 |
| Phú Nhuận | 10.801667 | 106.677500 |
| Tân Phú | 10.792222 | 106.625278 |
| Gò Vấp | 10.841667 | 106.666667 |
| Bình Tân | 10.771111 | 106.590556 |
| Thủ Đức | 10.849722 | 106.768333 |
| Bình Chánh | 10.750278 | 106.512500 |
| Nhà Bè | 10.651667 | 106.726389 |
| Hóc Môn | 10.878333 | 106.592500 |
| Củ Chi | 11.027778 | 106.483056 |
| Cần Giờ | 10.511944 | 106.880556 |

In [14]:
# Tọa độ thập phân theo khu vực (key là tên khu vực)
geo_decimal = {
    "Quận 1": (10.776111, 106.695833),
    "Quận 2": (10.778611, 106.757500),
    "Quận 3": (10.780000, 106.679444),
    "Quận 4": (10.761667, 106.702500),
    "Quận 5": (10.756667, 106.666667),
    "Quận 6": (10.746111, 106.636111),
    "Quận 7": (10.738611, 106.726389),
    "Quận 8": (10.723333, 106.627778),
    "Quận 9": (10.840000, 106.770833),
    "Quận 10": (10.773611, 106.667222),
    "Quận 11": (10.766944, 106.645556),
    "Quận 12": (10.861944, 106.658889),
    "Bình Thạnh": (10.802778, 106.696667),
    "Tân Bình": (10.803611, 106.650833),
    "Phú Nhuận": (10.801667, 106.677500),
    "Tân Phú": (10.792222, 106.625278),
    "Gò Vấp": (10.841667, 106.666667),
    "Bình Tân": (10.771111, 106.590556),
    "Thủ Đức": (10.849722, 106.768333),
    "Bình Chánh": (10.750278, 106.512500),
    "Nhà Bè": (10.651667, 106.726389),
    "Hóc Môn": (10.878333, 106.592500),
    "Củ Chi": (11.027778, 106.483056),
    "Cần Giờ": (10.511944, 106.880556),
    "center": (10.0039, 106.0044)
}

geo_decimal

{'Quận 1': (10.776111, 106.695833),
 'Quận 2': (10.778611, 106.7575),
 'Quận 3': (10.78, 106.679444),
 'Quận 4': (10.761667, 106.7025),
 'Quận 5': (10.756667, 106.666667),
 'Quận 6': (10.746111, 106.636111),
 'Quận 7': (10.738611, 106.726389),
 'Quận 8': (10.723333, 106.627778),
 'Quận 9': (10.84, 106.770833),
 'Quận 10': (10.773611, 106.667222),
 'Quận 11': (10.766944, 106.645556),
 'Quận 12': (10.861944, 106.658889),
 'Bình Thạnh': (10.802778, 106.696667),
 'Tân Bình': (10.803611, 106.650833),
 'Phú Nhuận': (10.801667, 106.6775),
 'Tân Phú': (10.792222, 106.625278),
 'Gò Vấp': (10.841667, 106.666667),
 'Bình Tân': (10.771111, 106.590556),
 'Thủ Đức': (10.849722, 106.768333),
 'Bình Chánh': (10.750278, 106.5125),
 'Nhà Bè': (10.651667, 106.726389),
 'Hóc Môn': (10.878333, 106.5925),
 'Củ Chi': (11.027778, 106.483056),
 'Cần Giờ': (10.511944, 106.880556),
 'center': (10.0039, 106.0044)}

In [15]:
vector_coridinates = {}
for key in geo_decimal:
    v = (geo_decimal[key][0] - geo_decimal["center"][0], geo_decimal[key][1] - geo_decimal["center"][1])
    manitude = (v[0]**2 + v[1]**2)**0.5
    if manitude == 0:
        vector_coridinates[key] = 0, 0
    else:
        vector_coridinates[key] = v[0] / manitude, v[1] / manitude
vector_coridinates

{'Quận 1': (0.7449980257653432, 0.6670666695359174),
 'Quận 2': (0.7170371291660058, 0.6970349742999794),
 'Quận 3': (0.7545215450151054, 0.656275276166959),
 'Quận 4': (0.7354693073807269, 0.6775580402451985),
 'Quận 5': (0.750796591415108, 0.6605334800897343),
 'Quận 6': (0.7615180878660177, 0.648143658345034),
 'Quận 7': (0.7132550543247266, 0.7009045780134636),
 'Quận 8': (0.7557569792919335, 0.6548521880940248),
 'Quận 9': (0.7371506268841126, 0.6757284612063931),
 'Quận 10': (0.7577613517856151, 0.6525317875322528),
 'Quận 11': (0.765606500372923, 0.6433091687413801),
 'Quận 12': (0.7951005092412601, 0.6064776831873445),
 'Bình Thạnh': (0.7557330601602844, 0.6548797918555528),
 'Tân Bình': (0.7776974662614158, 0.6286387285003795),
 'Phú Nhuận': (0.7642989942673554, 0.6448620374637579),
 'Tân Phú': (0.7856002629119874, 0.6187343750856388),
 'Gò Vấp': (0.7844854810231408, 0.6201471841941167),
 'Bình Tân': (0.7946250179273244, 0.6071005525314561),
 'Thủ Đức': (0.7421180303639201, 0.

In [16]:
def compute_cosine_similarity(vec1, vec2):
    dot_product = vec1[0] * vec2[0] + vec1[1] * vec2[1]
    magnitude_vec1 = (vec1[0]**2 + vec1[1]**2)**0.5
    magnitude_vec2 = (vec2[0]**2 + vec2[1]**2)**0.5
    if magnitude_vec1 == 0 or magnitude_vec2 == 0:
        return 0.0
    return dot_product / (magnitude_vec1 * magnitude_vec2)

def compare_two_time(time1, time2):
    return abs(time1 - time2) / (24 * 3600)

def compare_time_ranges(range1, range2):
    start_diff = compare_two_time(range1[0], range2[0])
    end_diff = compare_two_time(range1[1], range2[1])
    return (start_diff + end_diff) / 2

def compare_price(price1, price2):
    if price1 == 0 or price2 == 0:
        return 0.0
    return abs(price1 - price2) / max(price1, price2)



In [22]:
parsed_data = data.copy()
metakeyword_preprocess = []
print(type(preprocessed_store_infos))
for v in preprocessed_store_infos:
    metakeyword_preprocess.append(word2vec_model.wv.get_vector(v) if v in word2vec_model.wv.index_to_key else [0] * word2vec_model.vector_size)
parsed_data['MetaKeywords'] = metakeyword_preprocess

<class 'pandas.core.series.Series'>


ValueError: Length of values (103594) does not match length of index (107365)

In [18]:
parsed_data['Cuisines'] = cuisine_values
cuisine_embeddings = []
for values in cuisine_values:
    cuisine_embeddings.append(cuisine_model_bm25[cuisine_dictionary.doc2bow(values)])
parsed_data['CuisineEmbeddings'] = cuisine_embeddings


ValueError: Length of values (91204) does not match length of index (107365)

In [ ]:
districts_embeddings = []
for district in parsed_data['District']:
    if district in vector_coridinates:
        districts_embeddings.append(vector_coridinates[district])
    else:
        districts_embeddings.append((0, 0))
parsed_data['DistrictEmbeddings'] = districts_embeddings


In [ ]:

for label in ["Thứ hai", "Thứ ba", "Thứ tư", "Thứ năm", "Thứ sáu", "Thứ bảy", "Chủ nhật"]:
    time_embeddings = []
    for t in parsed_data[label]:
        
        if pd.isna(t):
            time_embeddings.append((0, 0))
        else:
            
            time_range = parse_time_point(t)
            time_embeddings.append(time_range)
    parsed_data[label + "_embeddings"] = time_embeddings
    